# 计算 LucaVirus 嵌入向量的相似度/距离矩阵

### 目的
本脚本加载指定文件夹 (`VECTOR_DIR`) 下的所有 LucaVirus 嵌入向量文件 (`.pt` 格式，内容为 NumPy 数组)，然后使用不同的距离度量（欧氏距离、曼哈顿距离、余弦距离）计算这些嵌入向量之间的成对距离矩阵。最后，计算并输出每个距离矩阵（压缩形式）的均值和标准差。

### 流程
1.  **导入库**：导入所需的 Python 库。
2.  **参数配置**：设置输入向量文件夹路径和要使用的距离度量列表。
3.  **加载嵌入向量**：遍历指定文件夹，加载 `.pt` 文件，提取 NumPy 数组。
4.  **数据准备与标准化**：将加载的数据转换为 NumPy 数组，并进行 Z-score 标准化（对欧氏距离和曼哈顿距离有意义，对余弦距离影响不大但保持一致性）。
5.  **计算距离矩阵**：循环遍历指定的距离度量：
    * 使用 `scipy.spatial.distance.pdist` 计算压缩形式的距离矩阵。
    * 计算该压缩矩阵的均值和标准差。
    * 打印结果。

### 1. 导入所需库

In [5]:
import os
import glob
import torch
import numpy as np
import pandas as pd # 虽然不用标签，但保留以备后用
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm

### 2. 参数配置

In [6]:
# --- 1. 输入向量文件夹路径 ---
VECTOR_DIR = "/data2/yangziyue/LucaVirus/data/251027_siqi/vector/" # <--- 修改为您的向量文件夹路径

# --- 2. 要使用的距离度量列表 ---
# 可选值参考 scipy.spatial.distance.pdist 文档, 例如:
# 'euclidean', 'cityblock' (曼哈顿), 'cosine', 'correlation', 'hamming', 'jaccard', etc.
METRICS = ['euclidean', 'cityblock', 'cosine']

print(f"向量文件夹: {VECTOR_DIR}")
print(f"将计算以下距离度量: {METRICS}")

向量文件夹: /data2/yangziyue/LucaVirus/data/251027_siqi/vector/
将计算以下距离度量: ['euclidean', 'cityblock', 'cosine']


### 3. 加载嵌入向量

In [9]:
print(f"\n正在查找 {VECTOR_DIR} 中的 .pt 文件...")
pt_files = glob.glob(os.path.join(VECTOR_DIR, "vector_*.pt"))
if not pt_files:
    print(f"错误: 在 {VECTOR_DIR} 中没有找到 'vector_*.pt' 格式的文件。")
    embeddings_list = [] # 确保变量存在
    # exit()
else:
    print(f"找到了 {len(pt_files)} 个 .pt 文件。")

embeddings_list = []
error_files = []
file_order = [] # 记录文件加载顺序

if pt_files: # 只有找到文件才继续
    print("\n正在加载 .pt 文件...")
    # 先对文件列表排序，确保每次加载顺序一致
    pt_files.sort()
    for pt_file in tqdm(pt_files):
        filename = os.path.basename(pt_file)
        try:
            # 加载 .pt 文件中的 NumPy 数组
            embedding_np = torch.load(pt_file, map_location=torch.device('cpu'))

            # 确保加载的是 NumPy 数组
            if not isinstance(embedding_np, np.ndarray):
                 print(f"警告: 文件 {filename} 内容不是 NumPy 数组，跳过。类型为 {type(embedding_np)}")
                 error_files.append(filename + " (类型错误)")
                 continue
            
            # 确保 embedding 是一个向量（一维数组）或可转换为向量
            if embedding_np.ndim == 1:
                embeddings_list.append(embedding_np)
                file_order.append(filename) # 记录成功加载的文件名
            elif embedding_np.ndim > 1 and embedding_np.shape[0] == 1:
                # print(f"警告: 文件 {filename} 的 embedding 不是一维 ({embedding_np.shape})，将使用第一个向量。")
                embeddings_list.append(embedding_np.flatten()) # 尝试展平
                file_order.append(filename)
            elif embedding_np.ndim == 0: # 处理标量情况
                 print(f"警告: 文件 {filename} 内容是标量，跳过。值: {embedding_np}")
                 error_files.append(filename + " (标量错误)")
                 continue
            else:
                # 对于更高维度或无法明确处理的情况
                # 尝试取第一个向量，如果不行则跳过
                try:
                    first_vector = embedding_np[0].flatten()
                    print(f"警告: 文件 {filename} 的 embedding 维度不符合预期 ({embedding_np.shape})，将尝试使用第一个向量展平后的结果。")
                    embeddings_list.append(first_vector)
                    file_order.append(filename)
                except:
                    print(f"警告: 文件 {filename} 的 embedding 维度 ({embedding_np.shape}) 无法处理，跳过。")
                    error_files.append(filename + " (维度错误)")

        except Exception as e:
            print(f"处理文件 {filename} 时出错: {e}")
            error_files.append(filename + f" ({e})")

    print(f"\n加载完成:")
    print(f"  成功加载的嵌入向量数量: {len(embeddings_list)}")
    if error_files:
         print(f"  警告: {len(error_files)} 个文件在加载或处理时发生错误:")
         print(f"    示例 (最多显示10个): {error_files[:10]}")

    if not embeddings_list:
        print("\n错误: 没有成功加载任何嵌入向量数据，无法继续。")
        # exit()
    else:
        # --- 4. 数据准备与标准化 ---
        print("\n正在将数据转换为 NumPy 数组...")
        X = np.array(embeddings_list)
        print(f"  嵌入向量矩阵 X 的维度: {X.shape}")

        print("\n正在对嵌入向量进行标准化 (StandardScaler)...")
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        print("标准化完成。")


正在查找 /data2/yangziyue/LucaVirus/data/251027_siqi/vector/ 中的 .pt 文件...
找到了 7000 个 .pt 文件。

正在加载 .pt 文件...


  0%|          | 0/7000 [00:00<?, ?it/s]


加载完成:
  成功加载的嵌入向量数量: 7000

正在将数据转换为 NumPy 数组...
  嵌入向量矩阵 X 的维度: (7000, 2560)

正在对嵌入向量进行标准化 (StandardScaler)...
标准化完成。


In [11]:
np.sum(X_scaled[:, 0])

0.0

### 5. 计算距离矩阵及统计量

In [8]:
if 'X_scaled' in locals() and X_scaled.shape[0] > 1: # 确保数据存在且至少有两个样本
    print("\n开始计算距离矩阵...")
    results = {}
    
    for metric in METRICS:
        print(f"-- 计算 {metric} 距离 --")
        try:
            # 使用 pdist 计算压缩距离矩阵 (只包含上三角部分的向量)
            # 注意：cosine 距离是 1 - cosine 相似度
            # 对于余弦距离，通常在原始数据上计算更有意义，但这里为了流程一致性，也在标准化数据上计算
            condensed_dist_matrix = pdist(X_scaled, metric=metric)
            
            # 计算均值和标准差
            mean_dist = np.mean(condensed_dist_matrix)
            std_dist = np.std(condensed_dist_matrix)
            
            results[metric] = {
                'mean': mean_dist,
                'std': std_dist,
                'condensed_matrix_shape': condensed_dist_matrix.shape
            }
            
            print(f"  压缩距离矩阵形状: {condensed_dist_matrix.shape}")
            print(f"  均值: {mean_dist:.6f}")
            print(f"  标准差: {std_dist:.6f}")
            
            # 可选：如果你需要完整的方阵形式
            # square_dist_matrix = squareform(condensed_dist_matrix)
            # print(f"  完整方阵形状: {square_dist_matrix.shape}")
            
        except ValueError as ve:
             print(f"  计算 {metric} 距离时出错: {ve}")
             print("  可能是因为数据包含 NaN 或 Inf 值，或者度量不适用于数据类型。")
             results[metric] = {'error': str(ve)}
        except Exception as e:
            print(f"  计算 {metric} 距离时发生未知错误: {e}")
            results[metric] = {'error': str(e)}
            
    print("\n--- 结果汇总 ---")
    for metric, stats in results.items():
        if 'error' in stats:
             print(f"度量: {metric:<12} | 错误: {stats['error']}")
        else:
             print(f"度量: {metric:<12} | 均值: {stats['mean']:.6f} | 标准差: {stats['std']:.6f}")

else:
    print("错误: 'X_scaled' 数据不存在或样本数不足 (需要至少 2 个)，请先运行前面的单元格。")

print("\n脚本执行完毕。")


开始计算距离矩阵...
-- 计算 euclidean 距离 --
  压缩距离矩阵形状: (24496500,)
  均值: 0.176301
  标准差: 0.104510
-- 计算 cityblock 距离 --
  压缩距离矩阵形状: (24496500,)
  均值: 6.341885
  标准差: 3.691088
-- 计算 cosine 距离 --
  压缩距离矩阵形状: (24496500,)
  均值: 0.000323
  标准差: 0.000308

--- 结果汇总 ---
度量: euclidean    | 均值: 0.176301 | 标准差: 0.104510
度量: cityblock    | 均值: 6.341885 | 标准差: 3.691088
度量: cosine       | 均值: 0.000323 | 标准差: 0.000308

脚本执行完毕。
